In [ ]:
!pip -q install torch torchvision transformers scikit-learn pillow tqdm accelerate torch pillow qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 17.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
path = '/content/drive/My Drive/work/research/other_misc/NakbaVirality/nakba_virality_train_with_images/'
# ===== Paths & Config (EDIT THESE) =====
# Prepared dataset CSV from prepare_dataset_60 notebook
CSV_PATH = path+"/train_df.csv"

# Where to write artifacts (predictions, submission, model)
OUT_DIR = path+"/Out"


In [ ]:
import os, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report



from transformers import AutoTokenizer, AutoProcessor, AutoModelForCausalLM
from qwen_vl_utils import process_vision_info

os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
base_dir = os.path.dirname(CSV_PATH)+'/images'

In [ ]:
len(os.listdir(base_dir))

1691

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

## LLaVA-OneVision-1.5-8B-Instruct

In [ ]:
model_path = "lmms-lab/LLaVA-OneVision-1.5-8B-Instruct"

# default: Load the model on the available device(s)
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype="auto", device_map="auto", trust_remote_code=True
)

# default processer
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)


config.json: 0.00B [00:00, ?B/s]

configuration_llavaonevision1_5.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/lmms-lab/LLaVA-OneVision-1.5-8B-Instruct:
- configuration_llavaonevision1_5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_rope_utils.py:927: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(


modeling_llavaonevision1_5.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/lmms-lab/LLaVA-OneVision-1.5-8B-Instruct:
- modeling_llavaonevision1_5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ImportError: cannot import name 'SlidingWindowCache' from 'transformers.cache_utils' (/usr/local/lib/python3.12/dist-packages/transformers/cache_utils.py)

In [ ]:
local_image_path = f"{base_dir}/{os.listdir(base_dir)[0]}"
print(local_image_path)

img = Image.open(local_image_path).convert("RGB")

propmpt_instruction='''
What is the chance that this image once publicised on local media, will be viral.
Answer 0 if it has low virality, 1 if it has medium virality and 2 if it has high virality. Give only the number, nothing else
'''
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": propmpt_instruction},
        ],
    }
]

# Build prompt
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Extract vision inputs
image_inputs, video_inputs = process_vision_info(messages)

# Prepare tensors
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)


inputs = {k: v.to(device) for k, v in inputs.items()}
model = model.to(device)

# Generate
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=512)

generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)

print(output_text[0])

In [ ]:
res=[]
for file_name in os.listdir(base_dir):
    local_image_path = f"{base_dir}/{file_name}"
    #print(local_image_path)

    img = Image.open(local_image_path).convert("RGB")

    propmpt_instruction='''
    What is the chance that this image once publicised on local media, will be viral.
    Answer 0 if it has low virality, 1 if it has medium virality and 2 if it has high virality. Give only the number, nothing else
    '''
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": propmpt_instruction},
            ],
        }
    ]

    # Build prompt
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Extract vision inputs
    image_inputs, video_inputs = process_vision_info(messages)

    # Prepare tensors
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )


    inputs = {k: v.to(device) for k, v in inputs.items()}
    model = model.to(device)

    # Generate
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    print(output_text[0])
    res.append({
        'ID':file_name,
        'ViralityLabel':output_text[0]
    })
    res_df=pd.DataFrame(res)
    res_df.to_csv(f'{OUT_DIR}/llava-1vision-8B.csv',index=False)

print(res_df.shape)


## Now try Molmo

In [ ]:
# predicts everything as 2

In [ ]:
print('done')

done


## Next we try Qwen 2.5

https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", device_map="auto"
)

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2.5-VL-7B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)


`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

["The image depicts a serene beach scene during what appears to be either sunrise or sunset, as indicated by the warm, golden light illuminating the sky and casting long shadows on the sand. A woman is sitting on the sandy beach, wearing a plaid shirt and dark pants, with her legs crossed. She has long hair and is smiling warmly at a light-colored dog, possibly a Labrador Retriever, which is sitting in front of her. The dog is wearing a harness and is extending its paw towards the woman's hand, suggesting a playful interaction. The ocean waves can be seen in the background, adding to the tranquil atmosphere of the"]


In [ ]:
PROMPT = """
You are a virality classifier for local news media.

Task:
Given an image, predict how likely it is to go viral if posted by local media.

Return ONLY one digit:
0 = low virality (ordinary, low emotion, low novelty, low shareability)
1 = medium virality (interesting/relatable, some novelty or emotion)
2 = high virality (very emotional, shocking, funny, dramatic, unusual, strong shareability)

Rules:
- Output must be exactly one character: 0 or 1 or 2
- No words, no punctuation, no explanation
"""

df=pd.read_csv(CSV_PATH)
df.head()


,ID,Text,image_name,ViralityLabel
0,1h1bwq4,Israel-Hezbollah permanent ceasefire has been ...,MiddleEast-1h1bwq4.jpg,Low viral
1,1.991948359108817e+18,"Plein soutien à l’AFPS de Villeneuve-d'Ascq, e...",1.991948359108817e_18.jpg,Medium viral
2,176e6bj,Iraq's Sistani urges end to Israel's horrific ...,Iraq-176e6bj.jpg,Medium viral
3,1oihf5a,"Israel launches strikes on Gaza, reports say, ...",Israel_Palestine-1oihf5a.jpeg,Medium viral
4,1.9486442584974136e+18,birleşmiş milletler işgal altındaki filistin t...,1.9486442584974136e_18.jpg,Medium viral


In [ ]:
results = []

all_files = sorted(os.listdir(base_dir))
counter=0
for index,row in df.iterrows():
    file_name=row['image_name']
    ID=row['ID']

    if counter%100==0:
        print(file_name)
        print(f'{counter}/{df.shape[0]}')
    counter+=1

    local_image_path = f"{base_dir}/{file_name}"
    img = Image.open(local_image_path).convert("RGB")



    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": img,
                },
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    # Preparation for inference
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")

    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    #print(int(output_text[0]))
    results.append({
        'ID':ID,
        'ViralityLabel':int(output_text[0])
    })
    res_df=pd.DataFrame(results)
    res_df.to_csv(f'{OUT_DIR}/Qwen2.5-VL-7B-Instruct.csv',index=False)





MiddleEast-1h1bwq4.jpg
0/1691
1.9568467501002468e_18.jpg
100/1691
1.9921106995863885e_18.jpg
200/1691
1.990742977069871e_18.jpg
300/1691
Gaza-1gh1x6v.jpg
400/1691
Syria-1j3jqv8.png
500/1691
1.7620256716923377e_18.jpg
600/1691
Palestine-1n2qgez.png
700/1691
1.7972577882565512e_18.jpg
800/1691
IsraelPalestine-1cpr48w.jpg
900/1691
IsraelPalestine-174cnyt.jpg
1000/1691
1.981413008732549e_18.jpg
1100/1691
1.9250518630422612e_18.jpg
1200/1691
IsraelPalestine-1ogr9gd.jpeg
1300/1691
MiddleEast-1iinw5c.jpg
1400/1691
1.995883649187697e_18.jpg
1500/1691
1.9994605181012296e_18.jpg
1600/1691


In [ ]:
print('done')

done


## Idefics2
https://huggingface.co/blog/idefics2

Cannot use it as every pred is 1

,ID,Text,image_name,ViralityLabel
0,1h1bwq4,Israel-Hezbollah permanent ceasefire has been ...,MiddleEast-1h1bwq4.jpg,Low viral
1,1.991948359108817e+18,"Plein soutien à l’AFPS de Villeneuve-d'Ascq, e...",1.991948359108817e_18.jpg,Medium viral
2,176e6bj,Iraq's Sistani urges end to Israel's horrific ...,Iraq-176e6bj.jpg,Medium viral
3,1oihf5a,"Israel launches strikes on Gaza, reports say, ...",Israel_Palestine-1oihf5a.jpeg,Medium viral
4,1.9486442584974136e+18,birleşmiş milletler işgal altındaki filistin t...,1.9486442584974136e_18.jpg,Medium viral


In [ ]:
import requests
import torch
from PIL import Image

from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image

DEVICE = "cuda:0"

# Note that passing the image urls (instead of the actual pil images) to the processor is also possible
image1 = load_image("https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg")
image2 = load_image("https://cdn.britannica.com/59/94459-050-DBA42467/Skyline-Chicago.jpg")
image3 = load_image("https://cdn.britannica.com/68/170868-050-8DDE8263/Golden-Gate-Bridge-San-Francisco.jpg")


processor = AutoProcessor.from_pretrained("HuggingFaceM4/idefics2-8b")
model = AutoModelForVision2Seq.from_pretrained(
    "HuggingFaceM4/idefics2-8b",
).to(DEVICE)


# Create inputs
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What do we see in this image?"},
        ]
    },
    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": "In this image, we can see the city of New York, and more specifically the Statue of Liberty."},
        ]
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "And how about this image?"},
        ]
    },
]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image1, image2], return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}


# Generate
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

print(generated_texts)


processor_config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Chat templates should be in a 'chat_template.jinja' file but found key='chat_template' in the processor's config. Make sure to move your template to its own file.


preprocessor_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model-00006-of-00007.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

model-00003-of-00007.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

['User: What do we see in this image? \nAssistant: In this image, we can see the city of New York, and more specifically the Statue of Liberty. \nUser: And how about this image? \nAssistant: In this image we can see buildings, trees, lights, water and sky.']


Do it for 1 image only

In [ ]:
PROMPT = """
You are a virality classifier for local news media.

Task:
Given an image, predict how likely it is to go viral if posted by local media.

Return ONLY one digit:
0 = low virality (ordinary, low emotion, low novelty, low shareability)
1 = medium virality (interesting/relatable, some novelty or emotion)
2 = high virality (very emotional, shocking, funny, dramatic, unusual, strong shareability)

Rules:
- Output must be exactly one character: 0 or 1 or 2
- No words, no punctuation, no explanation
"""

df=pd.read_csv(CSV_PATH)
df.head()


,ID,Text,image_name,ViralityLabel
0,1h1bwq4,Israel-Hezbollah permanent ceasefire has been ...,MiddleEast-1h1bwq4.jpg,Low viral
1,1.991948359108817e+18,"Plein soutien à l’AFPS de Villeneuve-d'Ascq, e...",1.991948359108817e_18.jpg,Medium viral
2,176e6bj,Iraq's Sistani urges end to Israel's horrific ...,Iraq-176e6bj.jpg,Medium viral
3,1oihf5a,"Israel launches strikes on Gaza, reports say, ...",Israel_Palestine-1oihf5a.jpeg,Medium viral
4,1.9486442584974136e+18,birleşmiş milletler işgal altındaki filistin t...,1.9486442584974136e_18.jpg,Medium viral


In [ ]:
import requests
import torch
from PIL import Image

from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image

DEVICE = "cuda:0"

# Note that passing the image urls (instead of the actual pil images) to the processor is also possible
image1 = load_image("https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg")




# Create inputs
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": PROMPT},
        ]
    }
]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image1], return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}


# Generate
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

print(generated_texts[0])


import re

def extract_virality_label(text: str):
    # find first standalone 0/1/2
    m = re.search(r"\b([012])\b", text)
    return int(m.group(1)) if m else None



User: 
You are a virality classifier for local news media.

Task:
Given an image, predict how likely it is to go viral if posted by local media.

Return ONLY one digit:
0 = low virality (ordinary, low emotion, low novelty, low shareability)
1 = medium virality (interesting/relatable, some novelty or emotion)
2 = high virality (very emotional, shocking, funny, dramatic, unusual, strong shareability)

Rules:
- Output must be exactly one character: 0 or 1 or 2
- No words, no punctuation, no explanation
 
Assistant: 1.


In [ ]:
answer = extract_virality_label(generated_texts[0][-20:])
print(answer)
answer = int(re.search(r"[012]", generated_texts[0][-20:]).group())
print(answer)


1
1


In [ ]:
results = []

all_files = sorted(os.listdir(base_dir))
counter=0
for index,row in df.iterrows():
    file_name=row['image_name']
    ID=row['ID']

    if counter%100==0:
        print(file_name)
        print(f'{counter}/{df.shape[0]}')
    counter+=1

    local_image_path = f"{base_dir}/{file_name}"
    img = Image.open(local_image_path).convert("RGB")



    # Create inputs
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT},
            ]
        }
    ]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image1], return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}


    # Generate
    generated_ids = model.generate(**inputs, max_new_tokens=500)
    generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

    #print(generated_texts[0])
    answer = extract_virality_label(generated_texts[0][-20:])


    #print(int(output_text[0]))
    results.append({
        'ID':ID,
        'ViralityLabel':int(answer)
    })
    res_df=pd.DataFrame(results)
    res_df.to_csv(f'{OUT_DIR}/Idefics2.csv',index=False)






MiddleEast-1h1bwq4.jpg
0/1691
1.9568467501002468e_18.jpg
100/1691
1.9921106995863885e_18.jpg
200/1691
1.990742977069871e_18.jpg
300/1691
Gaza-1gh1x6v.jpg
400/1691
Syria-1j3jqv8.png
500/1691
1.7620256716923377e_18.jpg
600/1691
Palestine-1n2qgez.png
700/1691
1.7972577882565512e_18.jpg
800/1691
IsraelPalestine-1cpr48w.jpg
900/1691
IsraelPalestine-174cnyt.jpg
1000/1691
1.981413008732549e_18.jpg
1100/1691
1.9250518630422612e_18.jpg
1200/1691
IsraelPalestine-1ogr9gd.jpeg
1300/1691
MiddleEast-1iinw5c.jpg
1400/1691
1.995883649187697e_18.jpg
1500/1691
1.9994605181012296e_18.jpg
1600/1691
